In [0]:
%sql
CREATE TABLE IF NOT EXISTS fk_1374.silver.products_scd1 (
    product_id STRING,
    product_name STRING,
    brand STRING,
    category_l1 STRING,
    category_l2 STRING,
    pack_size INT,
    pack_unit STRING,
    mrp DECIMAL(10,2),
    selling_price DECIMAL(10,2),
    is_active BOOLEAN,
    tags ARRAY<STRING>,
    last_changed_snapshot_date DATE
);

In [0]:
%sql
MERGE INTO fk_1374.silver.products_scd1 AS target
USING (
    SELECT
        product_id,
        product_name,
        brand,
        category_l1,
        category_l2,
        pack_size,
        pack_unit,
        CAST(mrp AS DECIMAL(10,2)) AS mrp,
        CAST(selling_price AS DECIMAL(10,2)) AS selling_price,
        is_active,
        tags,
        CAST(snapshot_date AS DATE) AS snapshot_date
    FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY product_id
                ORDER BY snapshot_date DESC
            ) AS rn
        FROM fk_1374.silver.products
    ) latest
    WHERE rn = 1
) AS source

ON target.product_id = source.product_id

-- Update only if at least one business column has changed
WHEN MATCHED AND (
       NOT (target.product_name <=> source.product_name)
    OR NOT (target.brand <=> source.brand)
    OR NOT (target.category_l1 <=> source.category_l1)
    OR NOT (target.category_l2 <=> source.category_l2)
    OR NOT (target.pack_size <=> source.pack_size)
    OR NOT (target.pack_unit <=> source.pack_unit)
    OR NOT (target.mrp <=> source.mrp)
    OR NOT (target.selling_price <=> source.selling_price)
    OR NOT (target.is_active <=> source.is_active)
    OR NOT (target.tags <=> source.tags)
)
THEN UPDATE SET
    target.product_name = source.product_name,
    target.brand = source.brand,
    target.category_l1 = source.category_l1,
    target.category_l2 = source.category_l2,
    target.pack_size = source.pack_size,
    target.pack_unit = source.pack_unit,
    target.mrp = source.mrp,
    target.selling_price = source.selling_price,
    target.is_active = source.is_active,
    target.tags = source.tags,
    target.last_changed_snapshot_date = source.snapshot_date

-- Insert new products
WHEN NOT MATCHED THEN INSERT (
    product_id,
    product_name,
    brand,
    category_l1,
    category_l2,
    pack_size,
    pack_unit,
    mrp,
    selling_price,
    is_active,
    tags,
    last_changed_snapshot_date
)
VALUES (
    source.product_id,
    source.product_name,
    source.brand,
    source.category_l1,
    source.category_l2,
    source.pack_size,
    source.pack_unit,
    source.mrp,
    source.selling_price,
    source.is_active,
    source.tags,
    source.snapshot_date
);